# 🏋️ Notebook 2 — Entraînement de TOUS les Modèles

**À lancer APRÈS le Notebook 1.**

## Bugs corrigés dans cette version

| Bug | Cause | Fix appliqué |
|-----|-------|--------------|
| `KeyboardInterrupt` sur Flair | Flair importe `tf_keras` qui bloque au chargement | Flair isolé dans sa propre cellule, importé APRÈS les autres |
| `evaluation_strategy` inconnu | Argument renommé dans transformers ≥ 4.41 | Détection automatique de version |
| `tokenizer=` dans Trainer | Argument renommé `processing_class=` | Détection automatique |

> ⚠️ Runtime → **GPU (A100)** recommandé avec Colab Pro (T4 also works)

In [1]:
# ── CELLULE 1 : Installation ──────────────────────────────────────────────────
# Flair est installé séparément car il tire tf_keras qui peut interférer
!pip install -q transformers datasets accelerate sentencepiece evaluate seqeval
!pip install -q matplotlib seaborn scikit-learn
# Flair: installer mais NE PAS importer ici
!pip install -q flair
print('✅ Installation terminée')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 23.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━

In [2]:
# ── CELLULE 2 : Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive monté')

Mounted at /content/drive
✅ Drive monté


In [3]:
# ── CELLULE 3 : Imports & Configuration ──────────────────────────────────────
# ⚠️  IMPORTANT : Flair n'est PAS importé ici — voir cellule dédiée plus bas
import json
import inspect
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch

warnings.filterwarnings('ignore')

# ─── CHEMINS ──────────────────────────────────────────────────────────────────
BASE_DIR   = '/content/drive/MyDrive/medical_project'
DATA_DIR   = f'{BASE_DIR}/datasets'
MODELS_DIR = f'{BASE_DIR}/models'

SEED     = 42
# A100 supporte bf16 nativement (plus stable que fp16)
_gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''
USE_BF16 = 'A100' in _gpu
USE_FP16 = torch.cuda.is_available() and not USE_BF16

# ─── Noms des modèles pré-entraînés ───────────────────────────────────────────
NER_MODEL_NAME = 'camembert-base'       # BERT français, token classification
QA_MODEL_NAME  = 'CATIE-AQ/QAmemberta' # Meilleur modèle QA français (benchmark CATIE)
CLF_MODEL_NAME = 'camembert-base'       # BERT français, classification de séquence

# ─── Hyperparamètres (optimisés pour A100 80GB — Colab Pro) ─────────────────
NER_EPOCHS  = 5
QA_EPOCHS   = 3
CLF_EPOCHS  = 5
BATCH_SIZE  = 32          # A100 80GB → x2 vs T4 (était 16)
LEARN_RATE  = 2e-5

print(f"GPU : {'✅ ' + torch.cuda.get_device_name(0) + (' [bf16]' if USE_BF16 else ' [fp16]') if torch.cuda.is_available() else '❌ CPU — changez le runtime !'}")

# ─── Helper : compatibilité entre versions de transformers ───────────────────
def get_eval_strategy_kwarg():
    """Retourne le bon nom d'argument selon la version de transformers installée."""
    from transformers import TrainingArguments
    params = inspect.signature(TrainingArguments.__init__).parameters
    return 'eval_strategy' if 'eval_strategy' in params else 'evaluation_strategy'

def get_trainer_tokenizer_kwarg():
    """Retourne 'processing_class' ou 'tokenizer' selon la version."""
    from transformers import Trainer
    params = inspect.signature(Trainer.__init__).parameters
    return 'processing_class' if 'processing_class' in params else 'tokenizer'

EVAL_STRAT_KEY   = get_eval_strategy_kwarg()
TRAINER_TOK_KEY  = get_trainer_tokenizer_kwarg()
print(f"eval key       : {EVAL_STRAT_KEY}")
print(f"trainer tok key: {TRAINER_TOK_KEY}")
print('✅ Config prête')

GPU : ❌ CPU — changez le runtime !
eval key       : eval_strategy
trainer tok key: processing_class
✅ Config prête


In [6]:
# ── CELLULE 4 : Chargement corpus BIO (partagé entre NER et Flair) ────────────
def load_bio_file(path: str) -> list:
    """Charge un fichier corpus BIO (token par ligne, ligne vide = fin de phrase)."""
    samples, tokens, tags = [], [], []
    for line in open(path, encoding='utf-8'):
        line = line.strip()
        if line:
            parts = line.split()
            if len(parts) == 2:
                tokens.append(parts[0])
                tags.append(parts[1])
        else:
            if tokens:
                samples.append({'tokens': tokens, 'tags': tags})
                tokens, tags = [], []
    if tokens:
        samples.append({'tokens': tokens, 'tags': tags})
    return samples

print('✅ Fonction load_bio_file prête')

✅ Fonction load_bio_file prête


In [7]:
# ════════════════════════════════════════════════════════════════════════════════
# MODÈLE 1A — CamemBERT NER
#
# Pourquoi CamemBERT ?
#   - Pré-entraîné sur du texte français (Wikipedia, CommonCrawl, etc.)
#   - Comprend la morphologie et le contexte grammatical français
#   - Fine-tuné pour la classification de tokens (B-ENTITE, I-ENTITE, O)
# Supérieur à : regex (pas de compréhension contextuelle), dictionnaires statiques
# ════════════════════════════════════════════════════════════════════════════════

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification, TrainingArguments, Trainer, set_seed,
)
import evaluate

set_seed(SEED)
print('\n' + '='*60)
print('MODÈLE 1A — CamemBERT NER')
print('='*60)

ner_dir = Path(DATA_DIR) / 'ner'
train_samples = load_bio_file(ner_dir / 'train.txt')
val_samples   = load_bio_file(ner_dir / 'dev.txt')
test_samples  = load_bio_file(ner_dir / 'test.txt')
print(f'  Corpus : {len(train_samples)} train | {len(val_samples)} val | {len(test_samples)} test')

# Vocabulaire des tags
all_tags  = sorted({tag for s in train_samples for tag in s['tags']})
label2id  = {tag: i for i, tag in enumerate(all_tags)}
id2label  = {i: tag for tag, i in label2id.items()}
print(f'  Tags ({len(all_tags)}) : {all_tags}')

ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL_NAME)

def tokenize_and_align(examples):
    """
    Tokenise les tokens et aligne les tags BIO avec les sous-tokens WordPiece.
    Les sous-tokens additionnels reçoivent -100 (ignoré par la loss CrossEntropy).
    """
    tokenized = ner_tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        padding='max_length',
        max_length=512,          # A100 80GB: can afford longer context
    )
    all_labels = []
    for i, tags in enumerate(examples['tags']):
        word_ids  = tokenized.word_ids(batch_index=i)
        prev_word = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)                  # [CLS], [SEP], [PAD]
            elif word_id != prev_word:
                label_ids.append(label2id[tags[word_id]])  # Premier sous-token
            else:
                label_ids.append(-100)                  # Sous-tokens suivants ignorés
            prev_word = word_id
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

def to_hf_ds(samples):
    return Dataset.from_dict({
        'tokens': [s['tokens'] for s in samples],
        'tags':   [s['tags']   for s in samples],
    })

print('  Tokenisation...')
ner_ds = DatasetDict({
    'train':      to_hf_ds(train_samples).map(tokenize_and_align, batched=True, remove_columns=['tokens','tags']),
    'validation': to_hf_ds(val_samples).map(tokenize_and_align,   batched=True, remove_columns=['tokens','tags']),
    'test':       to_hf_ds(test_samples).map(tokenize_and_align,  batched=True, remove_columns=['tokens','tags']),
})
print(f'  Train tokenisé : {len(ner_ds["train"])} features')

ner_model = AutoModelForTokenClassification.from_pretrained(
    NER_MODEL_NAME, num_labels=len(all_tags),
    id2label=id2label, label2id=label2id,
)

# Métriques : seqeval — standard CoNLL pour NER (F1 par entité)
seqeval_metric = evaluate.load('seqeval')

def compute_ner_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_preds  = [[id2label[pred] for pred, lbl in zip(preds, lbls) if lbl != -100]
                   for preds, lbls in zip(predictions, labels)]
    true_labels = [[id2label[lbl] for lbl in lbls if lbl != -100]
                   for lbls in labels]
    results = seqeval_metric.compute(predictions=true_preds, references=true_labels)
    out = {
        'precision': results['overall_precision'],
        'recall':    results['overall_recall'],
        'f1':        results['overall_f1'],
        'accuracy':  results['overall_accuracy'],
    }
    # F1 par entité
    for ent in ['MEDECIN','SPECIALITE','MEDICAMENT','GADGET']:
        if ent in results:
            out[f'f1_{ent}'] = results[ent]['f1']
    return out

ner_out = Path(MODELS_DIR) / 'camembert_ner'
ner_out.mkdir(parents=True, exist_ok=True)

ner_args = TrainingArguments(
    output_dir=str(ner_out),
    num_train_epochs=NER_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARN_RATE,
    weight_decay=0.01,
    **{EVAL_STRAT_KEY: 'epoch'},   # ← FIX: compatibilité version transformers
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    optim="adamw_torch",

    fp16=USE_FP16,
    bf16=USE_BF16,          # A100 native bf16
    tf32=True,              # A100 TensorFloat-32 cores
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    seed=SEED,
    report_to='none',
)

ner_trainer = Trainer(
    model=ner_model,
    args=ner_args,
    train_dataset=ner_ds['train'],
    eval_dataset=ner_ds['validation'],
    **{TRAINER_TOK_KEY: ner_tokenizer},   # ← FIX: processing_class ou tokenizer
    data_collator=DataCollatorForTokenClassification(ner_tokenizer),
    compute_metrics=compute_ner_metrics,
)

print('\n  🚀 Entraînement CamemBERT NER...')
ner_trainer.train()

ner_test_results = ner_trainer.evaluate(ner_ds['test'])
json.dump(ner_test_results, open(ner_out / 'test_results.json', 'w'), indent=2)
ner_trainer.save_model(str(ner_out / 'best_model'))
ner_tokenizer.save_pretrained(str(ner_out / 'best_model'))
json.dump({'labels': all_tags, 'label2id': label2id, 'id2label': id2label},
          open(ner_out / 'best_model' / 'labels.json', 'w'), indent=2)

print(f"  Test → F1: {ner_test_results.get('eval_f1',0):.4f} | "
      f"P: {ner_test_results.get('eval_precision',0):.4f} | "
      f"R: {ner_test_results.get('eval_recall',0):.4f}")
print(f'  ✅ Modèle sauvegardé → {ner_out}/best_model')

ner_history = ner_trainer.state.log_history
ner_results  = {'test': ner_test_results, 'history': ner_history}


MODÈLE 1A — CamemBERT NER
  Corpus : 4622 train | 1557 val | 1546 test
  Tags (8) : ['B-GADGET', 'B-MEDECIN', 'B-MEDICAMENT', 'B-SPECIALITE', 'I-GADGET', 'I-MEDECIN', 'I-SPECIALITE', 'O']
  Tokenisation...


Map:   0%|          | 0/4622 [00:00<?, ? examples/s]

Map:   0%|          | 0/1557 [00:00<?, ? examples/s]

Map:   0%|          | 0/1546 [00:00<?, ? examples/s]

Some weights of CamembertForTokenClassification were not initialized from the model checkpoint at camembert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Train tokenisé : 4622 features

  🚀 Entraînement CamemBERT NER...


RuntimeError: `fused=True` requires all the params to be floating point Tensors of supported devices: ['mps', 'cuda', 'xpu', 'hpu', 'cpu', 'mtia', 'privateuseone'] but torch.float32 and xla

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MODÈLE 1B — Flair NER  ← CELLULE ISOLÉE
#
# FIXES v5 — hyperparamètres corrigés après F1=0.01 :
#   - fine_tune=False  : LR unique pour LSTM (fine_tune=True nécessite deux
#                        groupes de LR séparés, non supporté dans ModelTrainer)
#   - learning_rate    : 0.1 → optimal pour SGD + LSTM seul (embeddings gelés)
#   - mini_batch_size  : 32  → meilleur gradient sur A100
#   - max_epochs       : 10  → convergence plus solide
#   - patience         : 5   → évite early stopping prématuré
#   - min_learning_rate: 1e-4 → valeur strictement < lr pour que anneal fonctionne
#   - anneal_factor    : 0.5
#   - Sauvegarde Drive : checkpoints désactivés pour économiser le quota Drive
# ════════════════════════════════════════════════════════════════════════════════

import torch, gc

if torch.cuda.is_available():
    torch.cuda.synchronize()
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU libéré — VRAM libre : {torch.cuda.memory_reserved()/1e9:.2f} GB réservés")

print('\n' + '='*60)
print('MODÈLE 1B — Flair NER (LSTM + CRF)')
print('='*60)
print('  Import Flair...')

try:
    from flair.data import Corpus
    from flair.datasets import ColumnCorpus
    from flair.embeddings import TransformerWordEmbeddings, StackedEmbeddings
    from flair.models import SequenceTagger
    from flair.trainers import ModelTrainer
    print('  ✅ Flair importé')

    flair_ner_dir = Path(DATA_DIR) / 'ner'
    flair_out_dir = Path(MODELS_DIR) / 'flair_ner'
    flair_out_dir.mkdir(parents=True, exist_ok=True)

    for fname in ['train.txt', 'dev.txt', 'test.txt']:
        fpath = flair_ner_dir / fname
        assert fpath.exists(), f"Fichier manquant : {fpath}"

    corpus: Corpus = ColumnCorpus(
        data_folder=flair_ner_dir,
        column_format={0: 'text', 1: 'ner'},
        train_file='train.txt',
        dev_file='dev.txt',
        test_file='test.txt',
    )
    print(f'  Corpus : {len(corpus.train)} train | {len(corpus.dev)} dev | {len(corpus.test)} phrases')

    TAG_TYPE = 'ner'
    tag_dict = corpus.make_label_dictionary(label_type=TAG_TYPE, add_unk=False)
    print(f'  Tags BIOES ({len(tag_dict)}) : {list(tag_dict.get_items())}')

    # ── Embeddings : fine_tune=False indispensable avec ModelTrainer ────────
    # ModelTrainer utilise SGD avec un seul LR global.
    # fine_tune=True nécessiterait un LR ~1000x plus faible pour le transformer
    # (1e-5 vs 0.1 pour le LSTM) — impossible avec SGD uniforme → loss stagnante.
    # Solution : geler CamemBERT, laisser LSTM + CRF apprendre librement.
    embeddings = StackedEmbeddings([
        TransformerWordEmbeddings(
            NER_MODEL_NAME,
            layers='-1,-2,-3,-4',     # Moyenne des 4 dernières couches = plus riche
            subtoken_pooling='first',
            fine_tune=False,           # ← CRITIQUE : SGD uniforme incompatible avec fine-tune
            use_context=True,
            use_context_separator=False,
        ),
    ])

    tagger = SequenceTagger(
        hidden_size=512,               # 512 > 256 : plus de capacité pour 4 entités
        embeddings=embeddings,
        tag_dictionary=tag_dict,
        tag_type=TAG_TYPE,
        use_crf=True,
        use_rnn=True,
        rnn_layers=2,                  # 2 couches BiLSTM
        dropout=0.1,
        locked_dropout=0.5,
        word_dropout=0.05,
    )

    flair_trainer = ModelTrainer(tagger, corpus)

    # ── SGD schedule optimal pour LSTM gelé sur embeddings ─────────────────
    # learning_rate=0.1  : valeur standard Flair pour SGD + LSTM
    # patience=5         : attend 5 epochs sans amélioration avant d'annealer
    # min_lr=1e-4        : doit être STRICTEMENT < learning_rate (sinon anneal bloqué)
    # anneal_factor=0.5  : divise le LR par 2 à chaque plateau
    flair_result = flair_trainer.train(
        base_path=str(flair_out_dir),
        learning_rate=0.1,
        mini_batch_size=32,
        max_epochs=10,
        patience=5,
        min_learning_rate=1e-4,
        anneal_factor=0.5,
        monitor_test=True,
        save_final_model=True,
        checkpoint=False,              # Désactivé : économise le quota Drive
    )

    # ── Lecture métriques ────────────────────────────────────────────────────
    flair_metrics = {}
    test_tsv = flair_out_dir / 'test.tsv'
    if test_tsv.exists():
        for line in open(test_tsv):
            if line.startswith('MICRO_AVG'):
                parts = line.strip().split('\t')
                if len(parts) >= 4:
                    flair_metrics = {'precision': float(parts[1]),
                                     'recall':    float(parts[2]),
                                     'f1':        float(parts[3])}

    if not flair_metrics:
        import re
        log_file = flair_out_dir / 'training.log'
        if log_file.exists():
            for line in open(log_file):
                if 'TEST' in line and 'f1' in line.lower():
                    nums = re.findall(r'\d+\.\d+', line)
                    if len(nums) >= 3:
                        flair_metrics = {'precision': float(nums[0]),
                                         'recall':    float(nums[1]),
                                         'f1':        float(nums[2])}

    print(f"  Test → F1: {flair_metrics.get('f1', 'N/A')} | "
          f"P: {flair_metrics.get('precision', 'N/A')} | "
          f"R: {flair_metrics.get('recall', 'N/A')}")
    print(f'  ✅ Flair NER sauvegardé → {flair_out_dir}')
    flair_ner_results = flair_metrics

except Exception as e:
    import traceback
    print(f'  ⚠️  Flair NER ignoré : {e}')
    traceback.print_exc()
    flair_ner_results = {}


In [ ]:
import torch
# ════════════════════════════════════════════════════════════════════════════════
# MODÈLE 2 — QAmemberta (QA Extractive) - VERSION CORRIGÉE
# ════════════════════════════════════════════════════════════════════════════════

from transformers import (
    AutoTokenizer, AutoModelForQuestionAnswering,
    DefaultDataCollator, TrainingArguments, Trainer, set_seed
)
from datasets import Dataset, DatasetDict
import gc
import json
import numpy as np
from pathlib import Path

# ⚠️ S'assurer que SEED et les autres variables sont définies
# Si vous exécutez cette cellule seule, décommentez les lignes ci-dessous :
# SEED = 42
# GPU flags (reprend la config de la cellule 3 si déjà définie, sinon détecte)
if 'USE_BF16' not in dir():
    _gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''
    USE_BF16 = 'A100' in _gpu
    USE_FP16 = torch.cuda.is_available() and not USE_BF16
BATCH_SIZE = 16
QA_EPOCHS = 3
LEARN_RATE = 2e-5
QA_MODEL_NAME = 'CATIE-AQ/QAmemberta'
DATA_DIR = '/content/drive/MyDrive/medical_project/datasets'
MODELS_DIR = '/content/drive/MyDrive/medical_project/models'
SEED = 42
print('\n' + '='*60)
print('MODÈLE 2 — QAmemberta (QA Extractive)')
print('='*60)

qa_dir   = Path(DATA_DIR) / 'qa'
qa_train = json.load(open(qa_dir / 'train.json',      encoding='utf-8'))
qa_val   = json.load(open(qa_dir / 'validation.json', encoding='utf-8'))
qa_test  = json.load(open(qa_dir / 'test.json',       encoding='utf-8'))
print(f'  QA chargé : {len(qa_train)} train | {len(qa_val)} val | {len(qa_test)} test')

qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
QA_MAX_LEN   = 384
QA_STRIDE    = 128

def preprocess_qa_batch(batch_examples, tokenizer, max_length, stride):
    """
    Tokenisation par batch avec gestion mémoire.
    """
    tokenized = tokenizer(
        batch_examples['question'],
        batch_examples['context'],
        truncation='only_second',
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length',
    )

    sample_map = tokenized.pop('overflow_to_sample_mapping')
    offset_mapping = tokenized.pop('offset_mapping')
    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answers = batch_examples['answers'][sample_idx]
        input_ids = tokenized['input_ids'][i]
        cls_idx = input_ids.index(tokenizer.cls_token_id) if tokenizer.cls_token_id in input_ids else 0

        # Cas sans réponse
        if not answers['answer_start'] or batch_examples['is_impossible'][sample_idx]:
            start_positions.append(cls_idx)
            end_positions.append(cls_idx)
            continue

        # Trouver le span de la réponse
        ans_start_char = answers['answer_start'][0]
        ans_end_char = ans_start_char + len(answers['text'][0])

        # Identifier les tokens du contexte (sequence_id == 1)
        seq_ids = tokenized.sequence_ids(i)
        ctx_start = None
        ctx_end = None
        for idx, sid in enumerate(seq_ids):
            if sid == 1:
                if ctx_start is None:
                    ctx_start = idx
                ctx_end = idx

        if ctx_start is None:
            start_positions.append(cls_idx)
            end_positions.append(cls_idx)
            continue

        # Vérifier que la réponse est dans le contexte tokenisé
        if offsets[ctx_start][0] > ans_start_char or offsets[ctx_end][1] < ans_end_char:
            start_positions.append(cls_idx)
            end_positions.append(cls_idx)
            continue

        # Trouver les tokens de début et fin
        start_token = ctx_start
        while start_token <= ctx_end and offsets[start_token][0] <= ans_start_char:
            start_token += 1
        start_token -= 1

        end_token = ctx_end
        while end_token >= ctx_start and offsets[end_token][1] >= ans_end_char:
            end_token -= 1
        end_token += 1

        start_positions.append(start_token)
        end_positions.append(end_token)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions'] = end_positions

    # Supprimer offset_mapping pour économiser la mémoire
    return {k: v for k, v in tokenized.items() if k != 'offset_mapping'}

def to_hf_qa(samples):
    return Dataset.from_list(samples)

print('  Tokenisation QA par lots (mémoire optimisée)...')

# Convertir en datasets HF
qa_ds_raw = DatasetDict({
    'train': to_hf_qa(qa_train),
    'validation': to_hf_qa(qa_val),
})

# Tokenisation avec map batch_size réduit
BATCH_SIZE_TOKENIZE = 32

qa_ds = DatasetDict({
    'train': qa_ds_raw['train'].map(
        lambda x: preprocess_qa_batch(x, qa_tokenizer, QA_MAX_LEN, QA_STRIDE),
        batched=True,
        batch_size=BATCH_SIZE_TOKENIZE,
        remove_columns=['question', 'context', 'answers', 'is_impossible'],
        load_from_cache_file=False
    ),
    'validation': qa_ds_raw['validation'].map(
        lambda x: preprocess_qa_batch(x, qa_tokenizer, QA_MAX_LEN, QA_STRIDE),
        batched=True,
        batch_size=BATCH_SIZE_TOKENIZE,
        remove_columns=['question', 'context', 'answers', 'is_impossible'],
        load_from_cache_file=False
    ),
})

# Libérer la mémoire des données brutes
del qa_ds_raw, qa_train, qa_val
gc.collect()

print(f'  Train tokenisé : {len(qa_ds["train"])} features')
print(f'  Val tokenisée  : {len(qa_ds["validation"])} features')

# Chargement du modèle
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)

# Gradient checkpointing désactivé sur A100 (VRAM suffisante, plus rapide sans)

qa_out = Path(MODELS_DIR) / 'qamemberta'
qa_out.mkdir(parents=True, exist_ok=True)

# Métrique de validation
def compute_qa_metrics(eval_pred):
    start_logits, end_logits = eval_pred.predictions
    start_labels, end_labels = eval_pred.label_ids

    start_preds = np.argmax(start_logits, axis=-1)
    end_preds = np.argmax(end_logits, axis=-1)

    return {
        'start_acc': float(np.mean(start_preds == start_labels)),
        'end_acc': float(np.mean(end_preds == end_labels)),
        'exact_acc': float(np.mean((start_preds == start_labels) & (end_preds == end_labels))),
    }

# ════════════════════════════════════════════════════════════════════════════════
# Arguments d'entraînement (version compatible avec transformers récent)
# ════════════════════════════════════════════════════════════════════════════════
qa_args = TrainingArguments(
    output_dir=str(qa_out),
    num_train_epochs=QA_EPOCHS,
    per_device_train_batch_size=16,          # A100 80GB: pas de contrainte OOM
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,           # batch effectif = 16 direct sur A100
    learning_rate=LEARN_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',                   # ← CORRECTION ici (au lieu de evaluation_strategy)
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='exact_acc',
    greater_is_better=True,
    optim="adamw_torch",

    fp16=USE_FP16,
    bf16=USE_BF16,          # A100 native bf16
    tf32=True,              # A100 TensorFloat-32
    seed=SEED,
    report_to='none',
    logging_steps=50,
    save_total_limit=2,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

qa_trainer = Trainer(
    model=qa_model,
    args=qa_args,
    train_dataset=qa_ds['train'],
    eval_dataset=qa_ds['validation'],
    tokenizer=qa_tokenizer,
    data_collator=DefaultDataCollator(),
    compute_metrics=compute_qa_metrics,
)
# Vérification GPU
print(f'  Modèle sur : {next(qa_model.parameters()).device}')

print('\n  🚀 Entraînement QAmemberta...')
print(f'     Batch size: {qa_args.per_device_train_batch_size}')
print(f'     Accumulation: {qa_args.gradient_accumulation_steps}')

gc.collect()

try:
    qa_trainer.train()
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('\n  ❌ ERREUR: Mémoire GPU insuffisante')
        print('  Solutions: réduire per_device_train_batch_size à 2 ou 4')
    raise

# Évaluation finale
qa_eval_results = qa_trainer.evaluate()
json.dump(qa_eval_results, open(qa_out / 'eval_results.json', 'w'), indent=2)
qa_trainer.save_model(str(qa_out / 'best_model'))
qa_tokenizer.save_pretrained(str(qa_out / 'best_model'))

print(f"\n  📊 Résultats validation:")
print(f"     exact_acc: {qa_eval_results.get('eval_exact_acc', 'N/A'):.4f}")
print(f'  ✅ QAmemberta sauvegardé → {qa_out}/best_model')

qa_history = qa_trainer.state.log_history
qa_results = {'test': qa_eval_results, 'history': qa_history}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MODÈLE 3 — CamemBERT Classification (×2 : type_visite + niveau_interet)
#
# Pourquoi Classification et non QA/NER pour ces champs ?
#   - type_visite : il n'existe pas de span textuel "premiere visite" dans le
#     compte-rendu — c'est une INTERPRÉTATION du contexte global → classification.
#   - niveau_interet : "3/5" est inféré du ton et du contenu global → idem.
# AutoModelForSequenceClassification lit tout le texte et prédit une classe.
# ════════════════════════════════════════════════════════════════════════════════

from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

def train_classifier(task_name: str, field: str) -> dict:
    """Entraîne un classificateur CamemBERT pour un champ catégoriel donné."""
    set_seed(SEED)
    print(f'\n{"="*60}')
    print(f'MODÈLE 3 — CamemBERT Classification : {task_name}')
    print('='*60)

    clf_data_dir = Path(DATA_DIR) / 'classification'
    train_df = pd.read_csv(clf_data_dir / f'{field}_train.csv').dropna()
    val_df   = pd.read_csv(clf_data_dir / f'{field}_val.csv').dropna()
    test_df  = pd.read_csv(clf_data_dir / f'{field}_test.csv').dropna()
    print(f'  Données : {len(train_df)} train | {len(val_df)} val | {len(test_df)} test')

    labels   = sorted(train_df['label'].unique())
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    print(f'  Classes ({len(labels)}) : {labels}')

    clf_tokenizer = AutoTokenizer.from_pretrained(CLF_MODEL_NAME)

    def encode_split(df):
        d = Dataset.from_dict({
            'text':  list(df['text']),
            'label': [label2id[l] for l in df['label']],
        })
        return d.map(
            lambda x: clf_tokenizer(x['text'], truncation=True,
                                    padding='max_length', max_length=512),  # A100: contexte plus long
            batched=True, remove_columns=['text'],
        )

    clf_ds = DatasetDict({
        'train':      encode_split(train_df),
        'validation': encode_split(val_df),
        'test':       encode_split(test_df),
    })

    clf_model = AutoModelForSequenceClassification.from_pretrained(
        CLF_MODEL_NAME, num_labels=len(labels),
        id2label=id2label, label2id=label2id,
    )

    accuracy_m = evaluate.load('accuracy')
    f1_m       = evaluate.load('f1')

    def compute_clf_metrics(p):
        preds  = np.argmax(p.predictions, axis=1)
        refs   = p.label_ids
        return {
            'accuracy': accuracy_m.compute(predictions=preds, references=refs)['accuracy'],
            'f1_macro': f1_m.compute(predictions=preds, references=refs, average='macro')['f1'],
        }

    clf_out = Path(MODELS_DIR) / task_name
    clf_out.mkdir(parents=True, exist_ok=True)

    clf_args = TrainingArguments(
        output_dir=str(clf_out),
        num_train_epochs=CLF_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARN_RATE,
        weight_decay=0.01,
        **{EVAL_STRAT_KEY: 'epoch'},   # ← FIX
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        optim="adamw_torch",

        fp16=USE_FP16,
        bf16=USE_BF16,          # A100 native bf16
        tf32=True,              # A100 TensorFloat-32
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        seed=SEED,
        report_to='none',
    )

    clf_trainer = Trainer(
        model=clf_model,
        args=clf_args,
        train_dataset=clf_ds['train'],
        eval_dataset=clf_ds['validation'],
        **{TRAINER_TOK_KEY: clf_tokenizer},   # ← FIX
        data_collator=DataCollatorWithPadding(clf_tokenizer),
        compute_metrics=compute_clf_metrics,
    )

    print(f'  🚀 Entraînement {task_name}...')
    clf_trainer.train()

    test_res = clf_trainer.evaluate(clf_ds['test'])
    json.dump(test_res, open(clf_out / 'test_results.json', 'w'), indent=2)
    clf_trainer.save_model(str(clf_out / 'best_model'))
    clf_tokenizer.save_pretrained(str(clf_out / 'best_model'))
    json.dump({'labels': labels, 'label2id': label2id, 'id2label': id2label},
              open(clf_out / 'best_model' / 'labels.json', 'w'), indent=2)

    print(f"  Test → Accuracy: {test_res.get('eval_accuracy',0):.4f} | "
          f"F1 macro: {test_res.get('eval_f1_macro',0):.4f}")
    print(f'  ✅ Modèle sauvegardé → {clf_out}/best_model')
    return {'test': test_res, 'history': clf_trainer.state.log_history}

tv_results = train_classifier('type_visite',    'type_visite')
ni_results = train_classifier('niveau_interet', 'niveau_interet')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# COURBES D'ÉVALUATION — Tableau de bord complet
# ════════════════════════════════════════════════════════════════════════════════

def _get(history, split, key):
    """Extrait (époques, valeurs) d'un historique Trainer."""
    full_key = f'{split}_{key}'
    eps, vals = [], []
    for e in history:
        if full_key in e and 'epoch' in e:
            eps.append(e['epoch'])
            vals.append(e[full_key])
    return eps, vals

# Regroupement des résultats
all_results = {
    'camembert_ner':  ner_results,
    'flair_ner':      {'test': flair_ner_results, 'history': []},
    'qamemberta':     qa_results,
    'type_visite':    tv_results,
    'niveau_interet': ni_results,
}

sns.set_theme(style='whitegrid', palette='muted')
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.suptitle("Courbes d'Entraînement — Pipeline Médical NLP",
             fontsize=15, fontweight='bold')

# Ligne 1 : CamemBERT NER
nh = ner_results.get('history', [])
for ax, (split, color, lbl) in zip(axes[0,:2], [
    ('train','#2196F3','Train'), ('eval','#FF9800','Val')
]):
    ep, vals = _get(nh, split, 'loss')
    if ep: ax.plot(ep, vals, 'o-', color=color, label=lbl, linewidth=2)
axes[0,0].set_title('CamemBERT NER — Loss') ; axes[0,0].legend()
ep, vf1 = _get(nh, 'eval', 'f1')
if ep:
    axes[0,1].plot(ep, vf1, 's-', color='#4CAF50', linewidth=2)
    axes[0,1].axhline(max(vf1), color='red', linestyle=':', label=f'Max F1={max(vf1):.3f}')
axes[0,1].set_title('CamemBERT NER — F1 Val') ; axes[0,1].set_ylim(0,1.05) ; axes[0,1].legend()
ep_p, vp = _get(nh,'eval','precision') ; ep_r, vr = _get(nh,'eval','recall')
if ep_p: axes[0,2].plot(ep_p, vp, '^-', color='#9C27B0', label='Precision', linewidth=2)
if ep_r: axes[0,2].plot(ep_r, vr, 'v-', color='#F44336', label='Recall',    linewidth=2)
axes[0,2].set_title('CamemBERT NER — P / R') ; axes[0,2].set_ylim(0,1.05) ; axes[0,2].legend()

# Ligne 2 : QAmemberta
qh = qa_results.get('history', [])
for ax, (split, color, lbl) in zip(axes[1,:2], [
    ('train','#2196F3','Train'), ('eval','#FF9800','Val')
]):
    ep, vals = _get(qh, split, 'loss')
    if ep: ax.plot(ep, vals, 'o-', color=color, label=lbl, linewidth=2)
axes[1,0].set_title('QAmemberta — Loss') ; axes[1,0].legend()
ep_e, ve = _get(qh, 'eval', 'exact_acc')
if ep_e: axes[1,1].plot(ep_e, ve, 's-', color='#00BCD4', linewidth=2)
axes[1,1].set_title('QAmemberta — Exact Accuracy') ; axes[1,1].set_ylim(0,1.05)
# Histogramme longueurs contextes QA
qa_train_data = json.load(open(Path(DATA_DIR)/'qa'/'train.json')) if (Path(DATA_DIR)/'qa'/'train.json').exists() else []
if qa_train_data:
    ctx_len = [len(s['context']) for s in qa_train_data[:2000]]
    axes[1,2].hist(ctx_len, bins=30, color='#00BCD4', alpha=0.8, edgecolor='white')
    axes[1,2].axvline(np.mean(ctx_len), color='red', linestyle='--', label=f'Moy={np.mean(ctx_len):.0f}')
    axes[1,2].legend()
axes[1,2].set_title('QA — Longueur des contextes')

# Ligne 3 : Classificateurs + Comparaison finale
for col, (task, color) in enumerate([('type_visite','#795548'),('niveau_interet','#607D8B')]):
    ch = all_results[task].get('history', [])
    for split, c, lbl in [('train','#2196F3','Train'),('eval','#FF9800','Val')]:
        ep, vals = _get(ch, split, 'loss')
        if ep: axes[2,col].plot(ep, vals, 'o-', color=c, label=lbl, linewidth=2)
    axes[2,col].set_title(f'{task} — Loss') ; axes[2,col].legend()

# Graphique récapitulatif
ax_s = axes[2,2]
models = ['CamemBERT NER','Flair NER','QAmemberta','CLF type','CLF interet']
f1_vals = [
    ner_results['test'].get('eval_f1', 0),
    flair_ner_results.get('f1', 0),
    1/(1 + qa_results['test'].get('eval_loss', 10)),  # loss inversée pour comparaison
    tv_results['test'].get('eval_f1_macro', 0),
    ni_results['test'].get('eval_f1_macro', 0),
]
colors_bar = ['#2196F3','#4CAF50','#00BCD4','#795548','#607D8B']
bars = ax_s.barh(models, f1_vals, color=colors_bar, edgecolor='white')
ax_s.set_xlim(0, 1.1)
ax_s.set_title('Comparaison finale (test set)')
for bar, val in zip(bars, f1_vals):
    if val > 0:
        ax_s.text(val+0.01, bar.get_y()+bar.get_height()/2,
                  f'{val:.3f}', va='center', fontweight='bold')

for row in axes:
    for ax in row:
        ax.set_xlabel('Époque')

plt.tight_layout()
curves_path = Path(MODELS_DIR) / 'training_curves.png'
plt.savefig(str(curves_path), dpi=150, bbox_inches='tight')
plt.show()
print(f'\n  ✅ Courbes sauvegardées → {curves_path}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# RÉSUMÉ FINAL
# ════════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('RÉSUMÉ DE L\'ENTRAÎNEMENT')
print('='*60)

for label, res_dict, metric in [
    ('CamemBERT NER  ',  ner_results['test'],  'eval_f1'),
    ('QAmemberta     ',  qa_results['test'],   'eval_exact_acc'),
    ('CLF type_visite',  tv_results['test'],   'eval_f1_macro'),
    ('CLF niveau_int ',  ni_results['test'],   'eval_f1_macro'),
]:
    val = res_dict.get(metric, None)
    val_str = f'{val:.4f}' if isinstance(val, float) else str(val)
    print(f'  {label} → {metric}: {val_str}')

if flair_ner_results:
    print(f'  Flair NER       → F1: {flair_ner_results.get("f1", "N/A")}')
else:
    print('  Flair NER       → non disponible')

print('='*60)
print('\n→ Lancer le Notebook 3 : pipeline d\'inférence sur vos rapports')